# 03 · Your first machine learning
The notebook version of [Level 5](../lessons/5-machine-learning/README.md).
Needs scikit-learn: run `uv sync --extra ml` once.

---

# 03 · はじめての機械学習
[レベル 5](../lessons/5-machine-learning/README.ja.md) のノートブック版です。
scikit-learn が必要です。`uv sync --extra ml` を 1 回実行しておきましょう。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from csi_lab import simulate_sequence
from csi_lab.features import window_features
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

## 1. Three data sets: train, test (same room), new room
Never test on the training data — that only shows what the model memorized.

---

## 1. 3 つのデータ: 学習用・テスト用（同じ部屋）・新しい部屋
学習に使ったデータでテストしてはいけません。それではモデルが暗記したかどうかしかわかりません。

In [ ]:
steps = [("empty", 60), ("still", 60), ("walk", 60), ("wave", 60)]

def make(seed, room_seed=7):
    w = window_features(simulate_sequence(steps, seed=seed, room_seed=room_seed))
    return w.features, w.labels, w.names

X_train, y_train, names = make(seed=1)
X_test, y_test, _ = make(seed=2)
X_room, y_room, _ = make(seed=2, room_seed=99)
print(names)
print(X_train.shape, "windows x features")

## 2. A decision tree you can read

---

## 2. 読める決定木

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(export_text(tree, feature_names=list(names)))
plt.figure(figsize=(12, 5))
plot_tree(tree, feature_names=list(names), class_names=list(tree.classes_), filled=True, fontsize=8);

## 3. Compare models

---

## 3. モデルをくらべる

In [ ]:
models = {
    "tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=0),
    "tree (no limit)": DecisionTreeClassifier(random_state=0),
    "random forest": RandomForestClassifier(random_state=0),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name:16s} train={accuracy_score(y_train, model.predict(X_train)):.3f} "
          f"test={accuracy_score(y_test, model.predict(X_test)):.3f} "
          f"new room={accuracy_score(y_room, model.predict(X_room)):.3f}")

## 4. Confusion matrix: where are the mistakes?
Rows = truth, columns = what the model said.

---

## 4. 混同行列: どこでまちがえた？
行 ＝ 本当の答え、列 ＝ モデルが答えたもの。

In [ ]:
labels = ["empty", "still", "walk", "wave"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_estimator(tree, X_test, y_test, labels=labels, ax=axes[0], colorbar=False)
axes[0].set_title("same room")
ConfusionMatrixDisplay.from_estimator(tree, X_room, y_room, labels=labels, ax=axes[1], colorbar=False)
axes[1].set_title("new room");

## 5. Your turn
- Remove the `spread` feature (`X[:, [0, 1, 3, 4]]`) and train again. Does the new room get better?
- Train on several rooms (`room_seed` = 7, 11, 23) together. Does that help?
- Can you invent a new feature that works in a new room? That is a real research question!

Report honestly: the simulator is cleaner than a real room.

---

## 5. やってみよう
- `spread` の特徴量を外して（`X[:, [0, 1, 3, 4]]`）もう一度学習させましょう。新しい部屋ではよくなりますか？
- いくつかの部屋（`room_seed` ＝ 7・11・23）をまとめて学習させましょう。役に立ちますか？
- 新しい部屋でも使える特徴量を考え出せますか？ これは本物の研究の問いです！

正直に報告しましょう。シミュレーターは本物の部屋よりきれいです。